# 05. Bowling Performance Analysis
**Project:** Sports Performance Analytics Using IPL Data  
**Domain:** Sports Analytics & Business Intelligence  

---

### Notebook Objective
1. Compute career bowling figures adhering to official cricket scoring rules:
   - Overs based solely on legal deliveries (`wides == 0 & noballs == 0`).
   - Runs conceded excluding byes and legbyes.
   - Wickets credited strictly to bowlers (bowled, caught, caught & bowled, lbw, stumped, hit wicket).
2. Calculate Economy Rate, Bowling Average, Bowling Strike Rate, and Dot Ball %.
3. Identify the Top 10 All-Time Wicket Takers in IPL history.
4. Generate quadrant scatter plot: Economy Rate vs Bowling Strike Rate.
5. Identify best single-match bowling spells.


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

clean_dir = os.path.join(PROJECT_DIR, 'Cleaned_Data')
viz_dir = os.path.join(PROJECT_DIR, 'Visualizations')

deliveries = pd.read_csv(os.path.join(clean_dir, 'deliveries.csv'), low_memory=False)


### 1. Feature Engineering Bowling Aggregates


In [2]:
df_bowl = deliveries[~deliveries['is_super_over']].copy()

match_bowl = df_bowl.groupby(['match_id', 'bowler']).agg(
    legal_balls=('is_legal_delivery', 'sum'),
    runs_conceded=('bowler_runs_conceded', 'sum'),
    wickets=('is_bowler_wicket', 'sum'),
    dots=('is_dot_ball', 'sum')
).reset_index()

bowling_stats = match_bowl.groupby('bowler').agg(
    matches=('match_id', 'nunique'),
    innings=('match_id', 'count'),
    legal_balls=('legal_balls', 'sum'),
    runs_conceded=('runs_conceded', 'sum'),
    wickets=('wickets', 'sum'),
    dot_balls=('dots', 'sum')
).reset_index().rename(columns={'bowler': 'player'})

bowling_stats['overs'] = ((bowling_stats['legal_balls'] // 6) + ((bowling_stats['legal_balls'] % 6) / 6.0)).round(1)
bowling_stats['overs_display'] = bowling_stats['legal_balls'].apply(lambda b: f"{b // 6}.{b % 6}")
bowling_stats['economy_rate'] = np.where(
    bowling_stats['legal_balls'] > 0,
    ((bowling_stats['runs_conceded'] * 6.0) / bowling_stats['legal_balls']).round(2),
    0.0
)
bowling_stats['bowling_average'] = np.where(
    bowling_stats['wickets'] > 0,
    (bowling_stats['runs_conceded'] / bowling_stats['wickets']).round(2),
    0.0
)
bowling_stats['bowling_strike_rate'] = np.where(
    bowling_stats['wickets'] > 0,
    (bowling_stats['legal_balls'] / bowling_stats['wickets']).round(2),
    0.0
)
bowling_stats['dot_ball_percentage'] = np.where(
    bowling_stats['legal_balls'] > 0,
    ((bowling_stats['dot_balls'] / bowling_stats['legal_balls']) * 100).round(2),
    0.0
)
bowling_stats = bowling_stats.sort_values(by=['wickets', 'economy_rate'], ascending=[False, True]).reset_index(drop=True)


### 2. Top 10 Wicket Takers Leaderboard


In [3]:
top10_bowl = bowling_stats.head(10)[['player', 'innings', 'overs_display', 'wickets', 'economy_rate', 'bowling_average', 'bowling_strike_rate', 'dot_ball_percentage']]
top10_bowl


,player,innings,overs_display,wickets,economy_rate,bowling_average,bowling_strike_rate,dot_ball_percentage
0,YS Chahal,184,672.5,233,8.05,23.25,17.33,32.67
1,B Kumar,206,766.4,226,7.71,26.16,20.35,41.24
2,SP Narine,200,775.1,207,6.79,25.42,22.47,38.36
3,PP Chawla,191,641.4,192,7.96,26.60,20.05,34.42
4,R Ashwin,217,785.0,187,7.20,30.22,25.19,34.10
5,JJ Bumrah,158,605.1,187,7.34,23.75,19.42,39.99
6,DJ Bravo,158,519.5,183,8.39,23.83,17.04,31.97
7,RA Jadeja,237,709.0,180,7.71,30.36,23.63,31.55
8,Rashid Khan,153,588.2,179,7.28,23.92,19.72,35.52
9,A Mishra,162,561.5,174,7.38,23.82,19.37,35.15


In [4]:
plt.figure(figsize=(10, 5))
top10_bplot = top10_bowl.sort_values(by='wickets', ascending=True)
bars = plt.barh(top10_bplot['player'], top10_bplot['wickets'], color='#d62728')
plt.title("Top 10 Wicket Takers in IPL History", fontsize=13, fontweight='bold')
plt.xlabel("Wickets Credited to Bowler")
plt.ylabel("Bowler")
for b in bars:
    w = b.get_width()
    plt.annotate(f"{int(w)}", xy=(w, b.get_y() + b.get_height()/2), xytext=(5, 0), textcoords="offset points", va='center')
plt.tight_layout()
plt.savefig(os.path.join(viz_dir, '05_top_10_wicket_takers.png'), dpi=200)
plt.show()


### 3. Bowling Economy vs Bowling Strike Rate Scatter Plot
Min 300 legal balls qualification threshold.


In [5]:
plt.figure(figsize=(9, 6))
qual_bowl = bowling_stats[bowling_stats['legal_balls'] >= 300].head(40)
scatter = plt.scatter(qual_bowl['economy_rate'], qual_bowl['bowling_strike_rate'], s=qual_bowl['wickets']*4, c=qual_bowl['dot_ball_percentage'], cmap='coolwarm', alpha=0.85)
plt.axvline(7.5, color='green', linestyle='--', label='Elite Economy (<= 7.5)')
plt.axhline(19.0, color='red', linestyle='--', label='Elite Strike Rate (<= 19.0)')
plt.title("Bowling Economy Rate vs Bowling Strike Rate (Min 300 Legal Balls)", fontsize=13, fontweight='bold')
plt.xlabel("Economy Rate (Runs / Over)")
plt.ylabel("Bowling Strike Rate (Balls / Wicket)")
plt.colorbar(scatter, label='Dot Ball %')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(viz_dir, '06_bowling_economy_vs_strike_rate.png'), dpi=200)
plt.show()


### 4. Best Individual Bowling Figures in a Match


In [6]:
best_spells = match_bowl.sort_values(by=['wickets', 'runs_conceded'], ascending=[False, True]).head(10)
best_spells[['bowler', 'wickets', 'runs_conceded', 'legal_balls', 'dots']].reset_index(drop=True)


### 5. Bowling Insights
- Yuzvendra Chahal, Dwayne Bravo, and Piyush Chawla lead the all-time wicket charts.
- Rashid Khan and Sunil Narine represent the gold standard of modern T20 bowling, maintaining an economy below 7.00 alongside exceptional strike rates.
- Alzarri Joseph's 6/12 and Sohail Tanvir's 6/14 remain the most destructive single-match spells recorded.
